# Investigation Template

Ad-hoc analysis skeleton for structured investigations. Supports four analysis types:
trend, comparison, distribution, and correlation.

**Outputs:** `investigation-{title}-{date}.json`, charts per analysis type

In [ ]:
# Papermill parameters
investigation_title = "ad-hoc-investigation"
bq_query = ""  # BigQuery SQL query (optional)
csv_paths = ""  # Comma-separated CSV file paths
analysis_type = "trend"  # trend | comparison | distribution | correlation
metric_columns = ""  # Comma-separated column names to analyse
date_column = ""  # Column to use as time axis (for trend analysis)
group_column = ""  # Column to group by (for comparison analysis)
report_date = "2026-04-04"
bq_project = None
output_dir = None

In [ ]:
import sys
from pathlib import Path

# Find workspace root (walk up to .git), then add data/notebooks/ to sys.path
# so that utils.pm_helpers is importable regardless of cwd
# (papermill runs notebooks from outputs/, not templates/)
_dir = Path.cwd()
while _dir != _dir.parent:
    if (_dir / ".git").exists():
        break
    _dir = _dir.parent
WORKSPACE_ROOT = _dir
NOTEBOOKS_DIR = WORKSPACE_ROOT / "data" / "notebooks"
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats
from utils.pm_helpers import (
    load_csv,
    detect_anomalies,
    set_chart_style,
    save_chart,
    save_json,
    BRAND_COLOURS,
    PALETTE,
)

set_chart_style()

metric_list = [m.strip() for m in metric_columns.split(",") if m.strip()] if metric_columns else []
csv_list = [p.strip() for p in csv_paths.split(",") if p.strip()] if csv_paths else []

print(f"Investigation: {investigation_title}")
print(f"Analysis type: {analysis_type}")
print(f"Metrics: {metric_list or 'auto-detect'}")

In [ ]:
# --- Load data ---

df = None
data_source = None

# Try BigQuery first
if bq_query:
    try:
        df = load_csv(bq_query, project=bq_project)
        data_source = "BigQuery"
        print(f"Loaded {len(df)} rows from BigQuery")
    except Exception as e:
        print(f"BigQuery failed: {e}")

# CSV fallback
if df is None and csv_list:
    frames = []
    for csv_path in csv_list:
        p = Path(csv_path) if Path(csv_path).is_absolute() else WORKSPACE_ROOT / csv_path
        if p.exists():
            chunk = pd.read_csv(p)
            chunk.columns = chunk.columns.str.strip().str.lower().str.replace(" ", "_")
            frames.append(chunk)
            print(f"Loaded: {csv_path} ({len(chunk)} rows)")
    if frames:
        df = pd.concat(frames, ignore_index=True)
        data_source = "CSV"

if df is not None:
    print(f"\nData source: {data_source}")
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    # Auto-detect metric columns if not specified
    if not metric_list:
        metric_list = [c for c in df.select_dtypes(include=[np.number]).columns[:6]]
        print(f"Auto-detected metrics: {metric_list}")
    display(df.head())
else:
    print("No data loaded. Provide bq_query or csv_paths.")

In [ ]:
# --- Analysis: Trend ---

findings = {"title": investigation_title, "type": analysis_type, "report_date": report_date, "findings": []}
chart_paths = []

if df is not None and analysis_type == "trend":
    # Determine date/time column
    time_col = date_column if date_column and date_column in df.columns else None
    if not time_col:
        for candidate in ["week_commencing", "date", "day", "week", "month", "period"]:
            if candidate in df.columns:
                time_col = candidate
                break

    for metric in metric_list:
        if metric not in df.columns:
            continue
        series = pd.to_numeric(df[metric], errors="coerce").dropna()
        if len(series) < 3:
            continue

        # Rolling average
        rolling_avg = series.rolling(window=min(4, len(series)), min_periods=2).mean()

        # Z-scores for changepoint detection
        z_scores = detect_anomalies(series, window=min(8, len(series)))
        anomalies_idx = z_scores[z_scores.abs() >= 2.0].index.tolist()

        finding = {
            "metric": metric,
            "trend": "increasing" if series.iloc[-1] > series.iloc[0] else "decreasing" if series.iloc[-1] < series.iloc[0] else "stable",
            "latest": float(series.iloc[-1]),
            "min": float(series.min()),
            "max": float(series.max()),
            "changepoints": len(anomalies_idx),
        }
        findings["findings"].append(finding)
        print(f"{metric}: {finding['trend']} (latest={finding['latest']:.2f}, {finding['changepoints']} changepoints)")

    # Chart: trend lines with rolling averages
    if metric_list:
        n_metrics = min(len([m for m in metric_list if m in df.columns]), 4)
        if n_metrics > 0:
            fig, axes = plt.subplots(1, n_metrics, figsize=(5 * n_metrics, 5))
            if n_metrics == 1:
                axes = [axes]
            for idx, metric in enumerate([m for m in metric_list if m in df.columns][:4]):
                ax = axes[idx]
                series = pd.to_numeric(df[metric], errors="coerce")
                ax.plot(range(len(series)), series, marker="o", linewidth=2, color=PALETTE[idx])
                rolling = series.rolling(window=min(4, len(series)), min_periods=2).mean()
                ax.plot(range(len(rolling)), rolling, linestyle="--", alpha=0.6, color=PALETTE[idx], label="Rolling avg")
                ax.set_title(metric.replace("_", " ").title())
                ax.legend()
            fig.suptitle(f"Trend Analysis — {investigation_title}", fontsize=14, fontweight="bold")
            plt.tight_layout()
            path = save_chart(fig, f"investigation-trend-{investigation_title}", report_date, output_dir)
            chart_paths.append(path)

elif df is not None and analysis_type == "comparison":
    grp = group_column if group_column and group_column in df.columns else None
    if grp:
        for metric in metric_list:
            if metric not in df.columns:
                continue
            grouped = df.groupby(grp)[metric].agg(["mean", "median", "std", "count"])
            for group_val, row in grouped.iterrows():
                findings["findings"].append({
                    "metric": metric,
                    "group": str(group_val),
                    "mean": float(row["mean"]),
                    "median": float(row["median"]),
                    "std": float(row["std"]) if pd.notna(row["std"]) else None,
                    "n": int(row["count"]),
                })
            print(f"\n{metric} by {grp}:")
            print(grouped.to_string())

        # Chart: grouped bar
        metric = [m for m in metric_list if m in df.columns][0] if metric_list else None
        if metric:
            fig, ax = plt.subplots(figsize=(10, 6))
            grouped_means = df.groupby(grp)[metric].mean().sort_values(ascending=False)
            ax.barh(grouped_means.index.astype(str), grouped_means.values, color=BRAND_COLOURS["primary_blue"], alpha=0.8)
            ax.set_xlabel(metric.replace("_", " ").title())
            ax.set_title(f"{metric} by {grp} — {investigation_title}")
            plt.tight_layout()
            path = save_chart(fig, f"investigation-comparison-{investigation_title}", report_date, output_dir)
            chart_paths.append(path)

elif df is not None and analysis_type == "distribution":
    for metric in metric_list:
        if metric not in df.columns:
            continue
        series = pd.to_numeric(df[metric], errors="coerce").dropna()
        q1, q3 = series.quantile(0.25), series.quantile(0.75)
        iqr = q3 - q1
        n_outliers = int(((series < q1 - 1.5 * iqr) | (series > q3 + 1.5 * iqr)).sum())
        # Shapiro-Wilk normality test (sample if large)
        sample = series.sample(min(5000, len(series)), random_state=42) if len(series) > 5000 else series
        _, p_normal = stats.shapiro(sample) if len(sample) >= 3 else (None, None)

        finding = {
            "metric": metric, "mean": float(series.mean()), "median": float(series.median()),
            "std": float(series.std()), "skew": float(series.skew()),
            "q1": float(q1), "q3": float(q3), "iqr": float(iqr),
            "n_outliers": n_outliers, "normality_p": float(p_normal) if p_normal else None,
        }
        findings["findings"].append(finding)
        print(f"{metric}: mean={finding['mean']:.2f}, median={finding['median']:.2f}, skew={finding['skew']:.2f}, outliers={n_outliers}")

    # Chart: histograms
    plot_metrics = [m for m in metric_list if m in df.columns][:4]
    if plot_metrics:
        fig, axes = plt.subplots(1, len(plot_metrics), figsize=(5 * len(plot_metrics), 5))
        if len(plot_metrics) == 1:
            axes = [axes]
        for idx, metric in enumerate(plot_metrics):
            ax = axes[idx]
            series = pd.to_numeric(df[metric], errors="coerce").dropna()
            ax.hist(series, bins=50, color=PALETTE[idx], alpha=0.7)
            ax.axvline(series.mean(), color=BRAND_COLOURS["red"], linestyle="--", label=f"Mean: {series.mean():.2f}")
            ax.axvline(series.median(), color=BRAND_COLOURS["green"], linestyle="--", label=f"Median: {series.median():.2f}")
            ax.set_title(metric.replace("_", " ").title())
            ax.legend(fontsize=8)
        fig.suptitle(f"Distribution Analysis — {investigation_title}", fontsize=14, fontweight="bold")
        plt.tight_layout()
        path = save_chart(fig, f"investigation-distribution-{investigation_title}", report_date, output_dir)
        chart_paths.append(path)

elif df is not None and analysis_type == "correlation":
    corr_cols = [m for m in metric_list if m in df.columns]
    if len(corr_cols) >= 2:
        corr_matrix = df[corr_cols].corr()
        for i, col_a in enumerate(corr_cols):
            for col_b in corr_cols[i+1:]:
                r = corr_matrix.loc[col_a, col_b]
                findings["findings"].append({
                    "pair": f"{col_a} vs {col_b}",
                    "correlation": round(float(r), 4),
                    "strength": "strong" if abs(r) > 0.7 else "moderate" if abs(r) > 0.4 else "weak",
                })
                print(f"{col_a} vs {col_b}: r={r:.3f}")

        # Chart: scatter matrix
        if len(corr_cols) <= 6:
            fig, axes = plt.subplots(len(corr_cols), len(corr_cols), figsize=(3 * len(corr_cols), 3 * len(corr_cols)))
            for i, col_a in enumerate(corr_cols):
                for j, col_b in enumerate(corr_cols):
                    ax = axes[i][j] if len(corr_cols) > 1 else axes
                    if i == j:
                        ax.hist(df[col_a].dropna(), bins=30, color=PALETTE[i % len(PALETTE)], alpha=0.7)
                    else:
                        ax.scatter(df[col_b], df[col_a], alpha=0.3, s=10, color=PALETTE[0])
                    if j == 0:
                        ax.set_ylabel(col_a[:15], fontsize=8)
                    if i == len(corr_cols) - 1:
                        ax.set_xlabel(col_b[:15], fontsize=8)
            fig.suptitle(f"Correlation Matrix — {investigation_title}", fontsize=14, fontweight="bold")
            plt.tight_layout()
            path = save_chart(fig, f"investigation-correlation-{investigation_title}", report_date, output_dir)
            chart_paths.append(path)

print(f"\n{len(findings['findings'])} findings, {len(chart_paths)} charts generated.")

In [ ]:
# --- Export ---

findings["data_source"] = data_source
findings["charts"] = chart_paths
save_json(findings, f"investigation-{investigation_title}", report_date, output_dir)

print(f"\n=== Investigation complete ===")
print(f"Title: {investigation_title}")
print(f"Type: {analysis_type}")
print(f"Findings: {len(findings['findings'])}")
print(f"Charts: {len(chart_paths)}")